In [3]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_validate
from  sklearn.ensemble import (
    GradientBoostingRegressor,
    HistGradientBoostingRegressor
)
from sklearn.metrics import r2_score

In [4]:
train_data = pd.read_csv("../data/processed_dataset.csv")
test_data = pd.read_parquet("../data/test_data.parquet")

X_train = train_data.drop(
    ["LoyerMensuel_Log1", "LoyerMensuel_BIF", "IdentifiantMaison"],
    axis=1).select_dtypes(include=["number"])
y_train = train_data["LoyerMensuel_Log1"]

print(X_train)

# correlations = X_train.corrwith(y_train).abs().sort_values(ascending=False)
# print(correlations.head(10))

     Chambres  Superficie_m2  DistanceRoute_m  AgeMaison  Salon_Bin  \
0         1.0      -1.591270         0.010545   0.040397          1   
1         4.0       0.701481         0.691637   0.306172          1   
2         4.0       0.218797        -1.084717   0.394764          1   
3         6.0       1.184165         1.223335  -1.554256          1   
4         3.0      -0.035954        -1.507659   1.280682          1   
..        ...            ...              ...        ...        ...   
377       4.0       0.245612         0.715805  -0.491154          1   
378       6.0       0.052373         1.162914   1.280682          1   
379       5.0       0.781928         1.682528  -0.313970          1   
380       6.0       1.640034        -1.568079   0.926315          1   
381       6.0       1.438915         0.981654   1.014907          1   

     SalleDeBainInterieure_Bin  Parking_Bin  Meuble_Bin  Jardin_Bin  \
0                            1            0           0           0   
1    

### Mise en niveau pour les données de test

In [6]:
# Les variables numériques
# feature_cols = ["AgeMaison", "Quartier_Target", "Indicateur_Confort", "Chambres_par_Superficie", "LoyerMensuel_Log1"]

# for col in ['Salon', 'SalleDeBainInterieure', 'Parking', 'Meuble', 'Jardin']:
#     test_data[col + "_Bin"] = test_data[col].map({"Oui": 1, "Non": 0}).fillna(0).astype(int)

# encoder = joblib.load("neighbourhood_encoder.joblib") 
# test_data["Quartier_Target"] = encoder.transform(test_data[["Quartier"]])[:, 0]

# test_data["LoyerMensuel_Log1"] = np.log1p(test_data["LoyerMensuel_BIF"])

# Séparation de X_test et y_test
X_test = test_data.drop(
    ["LoyerMensuel_Log1", "LoyerMensuel_BIF", "IdentifiantMaison"],
    axis=1).select_dtypes(include=["number"])
y_test = test_data["LoyerMensuel_Log1"]

X_test

,Chambres,Superficie_m2,DistanceRoute_m,AgeMaison,Salon_Bin,SalleDeBainInterieure_Bin,Parking_Bin,Meuble_Bin,Jardin_Bin,Quartier_Target
0,1.000000,-1.323112,1.767116,-1.377072,1,1,1,0,0,0.010428
1,2.000000,-0.491822,-1.036381,1.457866,1,1,1,0,0,0.001998
2,5.000000,1.264613,-1.459322,0.217581,1,1,0,0,1,0.010428
3,3.521649,1.130534,-0.130078,-1.465664,1,1,0,0,0,0.002163
4,1.000000,-1.698533,-0.093826,0.660540,0,1,1,0,0,0.002146
...,...,...,...,...,...,...,...,...,...,...
123,4.000000,0.580810,0.389536,0.571948,1,1,1,0,1,0.002029
124,5.000000,0.540586,-0.323423,-0.668338,1,1,1,0,1,0.001998
125,6.000000,1.197573,-0.480515,0.837723,1,1,1,0,1,0.002014
126,4.000000,0.178573,0.715805,-0.402562,1,1,0,1,1,0.002163


### Entrainement du modèle

In [10]:
boosting_model = GradientBoostingRegressor(
    learning_rate=0.01,
    n_estimators=162,
    max_depth=7,
    subsample=0.666,
    random_state=42
)

boosting_model.fit(X_train, y_train)

y_predict = boosting_model.predict(X_test)

y_predict_series = pd.Series(y_predict, index=y_test.index)

compareson = pd.DataFrame({
    "Valeurs réelles": np.expm1(y_test),
    "Valeurs prédites": np.expm1(y_predict_series)
})

np.expm1(y_predict[:5])

compareson.head(10)

,Valeurs réelles,Valeurs prédites
0,268868.0,4.720072e+05
1,1536788.0,1.139023e+06
2,889633.0,8.483135e+05
3,1657285.0,1.388857e+06
4,507985.0,3.666260e+05
5,2200592.0,1.712798e+06
6,296607.0,3.544999e+05
7,1537399.0,1.516999e+06
8,702286.0,8.308933e+05
9,1704211.0,1.540163e+06


### Mésure de performances avec cross_validate

In [11]:
scoring = ["neg_mean_squared_error", "neg_median_absolute_error", "neg_root_mean_squared_error", "r2"]

boosting_scores = cross_validate(boosting_model, X_train, y_train, scoring=scoring)

print(f"Fit Time : {boosting_scores["fit_time"]}")
print()
print(f"Score Time : {boosting_scores["score_time"]}")
print()
print(f"Test Neg Median Abosulte Error : {boosting_scores["test_neg_median_absolute_error"]}")
print()
print(f"Test Neg Mean Squared Error : {boosting_scores["test_neg_mean_squared_error"]}")
print()
print(f"Test R2 : {boosting_scores["test_r2"]}")

print(r2_score(y_test, y_predict))

Fit Time : [0.29414105 0.22148228 0.21653676 0.2256999  0.22656298]

Score Time : [0.00620937 0.00576663 0.00594091 0.00635648 0.00588727]

Test Neg Median Abosulte Error : [-0.20424892 -0.29710892 -0.24386227 -0.26661232 -0.27870464]

Test Neg Mean Squared Error : [-0.20455776 -0.17280816 -0.10799969 -0.09899362 -0.12178501]

Test R2 : [0.55157073 0.64932636 0.66077981 0.66568901 0.64313861]
0.7363747471553694


### Sauvegarde du modèle

In [9]:

mean = {
    col: X_train[col].mean() for col in X_train.select_dtypes(include=['number']).columns}
modes = {
    col: X_train[col].mode()[0] for col in X_train.select_dtypes(include=['object']).columns}

artefacts = {
    'boosting_model': boosting_model,
    'encoder': encoder,
    'mean': mean,
    'modes': modes,
    'columns': list(X_train.columns)
}

path = os.path.join(os.path.dirname(os.path.abspath(os.getcwd())), "data", "best_model_tuned.pkl")
joblib.dump(artefacts, path)

print(f"Modèle et prétraitements sauvegardés dans {path}")

NameError: name 'encoder' is not defined

In [8]:
importance = boosting_model.feature_importances_

print(X_train.columns)
print(importance)

classement = pd.DataFrame({
    "Variables": X_train.columns,
    "Feature_importance": importance
}).sort_values(ascending=False, by="Feature_importance")

classement

Index(['Chambres', 'Superficie_m2', 'DistanceRoute_m', 'AgeMaison',
       'Salon_Bin', 'SalleDeBainInterieure_Bin', 'Parking_Bin', 'Meuble_Bin',
       'Jardin_Bin', 'Quartier_Target'],
      dtype='str')
[0.21780769 0.29959394 0.01838818 0.01657394 0.01449356 0.00696379
 0.00324737 0.04131066 0.02018849 0.36143237]


,Variables,Feature_importance
9,Quartier_Target,0.361432
1,Superficie_m2,0.299594
0,Chambres,0.217808
7,Meuble_Bin,0.041311
8,Jardin_Bin,0.020188
2,DistanceRoute_m,0.018388
3,AgeMaison,0.016574
4,Salon_Bin,0.014494
5,SalleDeBainInterieure_Bin,0.006964
6,Parking_Bin,0.003247
